# RAG — Retrieval Augmented Generation

## Jak sprawić, żeby LLM "wiedział" o rzeczach, których nie ma w jego danych treningowych?

### Plan warsztatu

1. **Problem**: Pokażemy, że lokalny LLM *nie zna* szczegółów o prezydentach Polski
2. **Embeddingi**: Zamienimy tekst o prezydentach na wektory liczbowe (embeddingi)
3. **Wyszukiwanie semantyczne**: Znajdziemy najbardziej trafne fragmenty tekstu dla danego pytania
4. **RAG**: Skleimy znalezione fragmenty z pytaniem i wyślemy do LLM-a — i zobaczymy, że teraz odpowie poprawnie!

### Czym jest RAG?

**Retrieval Augmented Generation** to technika, w której:
- Mamy bazę dokumentów (pliki tekstowe, PDF-y, strony www...)
- Zamieniamy je na embeddingi (wektory liczbowe) — to już widzieliśmy przy Word2Vec!
- Gdy użytkownik zadaje pytanie, szukamy najlepiej pasujących fragmentów
- Te fragmenty doklejamy do pytania i wysyłamy do LLM-a
- LLM odpowiada na podstawie dostarczonych fragmentów — nie "halucynuje"!

To jest sposób w jaki deweloperzy budują chatboty firmowe, asystentów dokumentacji, systemy Q&A itp.

## 1. Konfiguracja środowiska

In [1]:
# === Instalacja i import pakietów ===
import subprocess, sys

def ensure_package(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"Instaluję {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("sentence-transformers", "sentence_transformers")
ensure_package("openai")
ensure_package("numpy")

import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import requests
import json
import textwrap

print("Wszystkie pakiety załadowane!")

Wszystkie pakiety załadowane!


## 2. Połączenie z lokalnym LLM-em (Ollama)

Poniższa komórka automatycznie:
1. Sprawdzi czy masz uruchomione **Ollama** na swoim komputerze
2. Jeśli tak — wybierze najlepszy dostępny model
3. Jeśli nie — połączy się z serwerem prowadzącego

### Jak zainstalować Ollama (jeśli chcesz uruchomić lokalnie):
1. Wejdź na https://ollama.ai i pobierz dla swojego systemu
2. Zainstaluj i uruchom
3. W terminalu wpisz: `ollama pull qwen2.5:7b` (lub inny model)
4. Ollama działa w tle i wystawia API na `http://localhost:11434`

In [ ]:
# === Auto-detekcja modelu LLM ===

# Preferowane modele (od najlepszego do najmniejszego)
PREFERRED_MODELS = [
    # Najlepsze modele
    "qwen3.5:27b", "qwen3.5:9b", "qwen3.5:4b",
    "qwen3:14b", "qwen3:8b",
    "gemma4:12b", "gemma4:4b",
    "gemma3:27b", "gemma3:12b", "gemma3:8b",
    "qwen2.5:14b", "qwen2.5:7b",
    "gemma2:9b", "mistral:7b", "llama3.1:8b",
    # Małe modele
    "qwen3.5:2b", "qwen3.5:0.8b",
    "qwen3:4b", "qwen3:1.7b", "qwen3:0.6b",
    "gemma4:e4b", "gemma4:e2b",
    "gemma3:4b", "gemma3:1b",
    "qwen2.5:3b",
]

INSTRUCTOR_SERVER = "http://192.168.0.142:1234"  # <-- Adres serwera prowadzącego

def detect_ollama(base_url="http://localhost:11434"):
    headers = {"ngrok-skip-browser-warning": "true"} if "ngrok" in base_url else {}
    try:
        r = requests.get(f"{base_url}/api/tags", timeout=3, headers=headers)
        if r.status_code == 200:
            return [m["name"] for m in r.json().get("models", [])]
    except:
        pass
    return []

def detect_lmstudio(base_url="http://localhost:1234"):
    try:
        r = requests.get(f"{base_url}/v1/models", timeout=3)
        if r.status_code == 200:
            models = r.json().get("data", [])
            return [m["id"] for m in models]
    except:
        pass
    return []

def pick_best_model(available_models):
    for preferred in PREFERRED_MODELS:
        pname = preferred.split(":")[0]
        for available in available_models:
            if pname in available:
                return available
    return available_models[0] if available_models else None

OLLAMA_URL = None
MODEL_NAME = None

print("Szukam LM Studio (port 1234)...")
lms_models = detect_lmstudio("http://localhost:1234")
if lms_models:
    OLLAMA_URL = "http://localhost:1234"
    MODEL_NAME = pick_best_model(lms_models) or lms_models[0]
    print(f"✓ LM Studio znalezione! Wybrany model: {MODEL_NAME}")
else:
    print("  LM Studio niedostępne.")
    print("Szukam lokalnej Ollamy (port 11434)...")
    local_models = detect_ollama("http://localhost:11434")
    if local_models:
        OLLAMA_URL = "http://localhost:11434"
        MODEL_NAME = pick_best_model(local_models)
        print(f"✓ Lokalna Ollama znaleziona! Wybrany model: {MODEL_NAME}")
    else:
        print("  Ollama niedostępna.")
        print("Próbuję serwer prowadzącego...")
        instructor_models = detect_lmstudio(INSTRUCTOR_SERVER)
        if instructor_models:
            OLLAMA_URL = INSTRUCTOR_SERVER
            MODEL_NAME = instructor_models[0]
            print(f"✓ Serwer prowadzącego dostępny! Wybrany model: {MODEL_NAME}")
        else:
            print("✗ Brak dostępnego LLM-a! Zainstaluj Ollama lub LM Studio, lub poproś o adres serwera.")

if OLLAMA_URL:
    is_lmstudio = "1234" in OLLAMA_URL
    base = f"{OLLAMA_URL}/v1"  # LM Studio i Ollama — oba używają /v1
    client = OpenAI(base_url=base, api_key="lm-studio" if is_lmstudio else "ollama")
    print(f"\nKlient LLM gotowy! ({'LM Studio' if is_lmstudio else 'Ollama'})")

In [ ]:
# === Inicjalizacja klienta OpenAI (kompatybilny z Ollama) ===

if OLLAMA_URL:
    client = OpenAI(
        base_url=f"{OLLAMA_URL}/v1",
        api_key="lm-studio" if "1234" in OLLAMA_URL else "ollama",
    )
    print(f"Klient LLM gotowy! Model: {MODEL_NAME}")
    print(f"Serwer: {OLLAMA_URL}")
else:
    print("Klient LLM niedostępny. Przejdź do sekcji o embeddingach — ta część zadziała bez LLM-a.")

## 3. Problem: LLM nie zna odpowiedzi

Zapytajmy nasz lokalny model o prezydentów Polski.
Mniejsze modele (7B parametrów) zazwyczaj **nie znają** dobrze polskiej historii — 
szczególnie szczegółów jak daty kadencji czy okoliczności odejścia z urzędu.

Zobaczmy co się stanie:

In [ ]:
# === Pytanie do "gołego" LLM-a (bez RAG) ===

TEST_QUESTION = "Który prezydent Polski sprawował urząd najkrócej i dlaczego?"

if OLLAMA_URL:
    print(f"Pytanie: {TEST_QUESTION}")
    print(f"Model: {MODEL_NAME}")
    print("="*60)
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Odpowiadaj krótko i po polsku."},
            {"role": "user", "content": TEST_QUESTION}
        ],
        temperature=0.1,  # Niska temperatura = bardziej deterministyczna odpowiedź
    )
    
    answer = response.choices[0].message.content
    print(f"\nOdpowiedź LLM-a (BEZ RAG):\n")
    print(textwrap.fill(answer, width=80))
    print("\n" + "="*60)
    print("\nCzy odpowiedź jest poprawna? Sprawdźmy później z RAG-iem!")
else:
    print("LLM niedostępny — przejdź dalej do embeddingów.")

## 4. Przygotowanie dokumentów

Wczytujemy plik tekstowy o prezydentach Polski. 
Następnie dzielimy go na mniejsze **fragmenty (chunks)** — 
to kluczowy krok, bo embedding jednego ogromnego tekstu nie da dobrych wyników przy wyszukiwaniu.

### Dlaczego dzielimy na fragmenty?
- Modele embeddingowe najlepiej działają na krótszych tekstach (1-3 akapity)
- Chcemy wyszukać **konkretny** fragment odpowiadający na pytanie, nie cały dokument
- To samo robią profesjonalne systemy RAG (LangChain, LlamaIndex, itd.)

In [ ]:
# === Wczytanie dokumentu ===

with open("prezydenci_polski.txt", "r", encoding="utf-8") as f:
    full_text = f.read()

print(f"Wczytano dokument: {len(full_text)} znaków")
print(f"Pierwsze 300 znaków:\n")
print(full_text[:300] + "...")

In [ ]:
# === Podział na fragmenty (chunking) ===

def chunk_text(text, chunk_size=300, overlap=50):
    """
    Dzieli tekst na nakładające się fragmenty.
    
    Args:
        text: tekst do podziału
        chunk_size: docelowy rozmiar fragmentu (w znakach)
        overlap: ile znaków nakładania między fragmentami
    
    Dlaczego overlap? Żeby nie "przeciąć" zdania w połowie
    i nie stracić kontekstu na granicy fragmentów.
    """
    # Dzielimy po podwójnych enterach (akapitach)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    
    chunks = []
    current_chunk = ""
    
    for para in paragraphs:
        if len(current_chunk) + len(para) < chunk_size:
            current_chunk += (" " if current_chunk else "") + para
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = para
    
    if current_chunk:
        chunks.append(current_chunk)
    
    return chunks

chunks = chunk_text(full_text, chunk_size=500, overlap=50)

print(f"Dokument podzielony na {len(chunks)} fragmentów\n")
for i, chunk in enumerate(chunks):
    print(f"--- Fragment {i+1} ({len(chunk)} znaków) ---")
    print(textwrap.fill(chunk[:150], width=80) + ("..." if len(chunk) > 150 else ""))
    print()

### Ćwiczenie 1: Eksperymentuj z rozmiarem fragmentów

Spróbuj zmienić parametry `chunk_size` i `overlap` w komórce powyżej.

**Proszę uzupełnić poniższą komórkę** — uruchom chunking z innymi parametrami i porównaj wyniki:

In [ ]:
# === Ćwiczenie 1 ===
# Spróbuj różnych rozmiarów fragmentów i zobacz jak to wpływa na liczbę chunków

### Proszę poniżej uzupełnić kod ### (≈ 2 linijki)
# Wskazówka: wywołaj chunk_text z innymi parametrami, np. chunk_size=200 lub chunk_size=1000

chunks_male = chunk_text(full_text, chunk_size=200, overlap=30)    # małe fragmenty
chunks_duze = chunk_text(full_text, chunk_size=1000, overlap=100)   # duże fragmenty

### Koniec uzupełniania kodu ###

print(f"Domyślne (500 znaków): {len(chunks)} fragmentów")
print(f"Małe (200 znaków):     {len(chunks_male)} fragmentów")
print(f"Duże (1000 znaków):    {len(chunks_duze)} fragmentów")

print(f"\nPrzykładowy mały fragment:")
print(f"  '{chunks_male[0][:100]}...'")
print(f"\nPrzykładowy duży fragment:")
print(f"  '{chunks_duze[0][:100]}...'")
print(f"\nJak myślisz — lepiej mieć więcej małych fragmentów czy mniej dużych? Dlaczego?")

## 5. Embeddingi — zamiana tekstu na wektory

Pamiętacie **Word2Vec** z poprzednich zajęć? Tam każde *słowo* miało swój wektor.
Teraz robimy to samo, ale dla **całych fragmentów tekstu** — to są tzw. *sentence embeddings*.

Używamy modelu `paraphrase-multilingual-MiniLM-L12-v2` — lekkiego modelu (do uruchomienia na CPU),
który rozumie wiele języków, w tym **polski**.

### Jak to działa?
```
"Prezydent Kaczyński zginął w katastrofie" → [0.12, -0.45, 0.78, ..., 0.03]  (384 liczby)
```

Teksty o podobnym *znaczeniu* będą miały **bliskie wektory** (mały kąt między nimi = wysokie cosine similarity).

In [ ]:
# === Ładowanie modelu embeddingowego ===
# Ten model działa na CPU — nie potrzeba GPU!
# Przy pierwszym uruchomieniu zostanie pobrany (~120 MB)

print("Ładowanie modelu embeddingowego...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print(f"Model załadowany!")
print(f"Wymiar embeddingów: {embed_model.get_sentence_embedding_dimension()}")

In [ ]:
# === Tworzenie embeddingów dla naszych fragmentów ===

print(f"Tworzę embeddingi dla {len(chunks)} fragmentów...")
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True)

print(f"\nGotowe! Kształt macierzy embeddingów: {chunk_embeddings.shape}")
print(f"Każdy fragment to wektor {chunk_embeddings.shape[1]} liczb\n")

# Zobaczmy jak wygląda embedding pierwszego fragmentu
print(f"Embedding fragmentu 1 (pierwsze 10 wartości):")
print(np.round(chunk_embeddings[0][:10], 4))

### Ćwiczenie 2: Cosine similarity "ręcznie"

Zanim użyjemy gotowej funkcji wyszukiwania, policzmy cosine similarity sami.

Przypomnijmy wzór:

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

Gdzie $A \cdot B$ to iloczyn skalarny, a $\|A\|$ to norma (długość) wektora.

In [ ]:
# === Ćwiczenie 2: Policz cosine similarity ręcznie ===

# Weźmy embedding pytania i pierwszego fragmentu
pytanie = "Kto był najmłodszym prezydentem?"
emb_pytania = embed_model.encode([pytanie])[0]
emb_fragment_0 = chunk_embeddings[0]
emb_fragment_1 = chunk_embeddings[1]

### Proszę poniżej uzupełnić kod ### (≈ 3 linijki)
# Wskazówka: użyj np.dot() do iloczynu skalarnego i np.linalg.norm() do normy

similarity_0 = np.dot(emb_pytania, emb_fragment_0) / (np.linalg.norm(emb_pytania) * np.linalg.norm(emb_fragment_0))
similarity_1 = np.dot(emb_pytania, emb_fragment_1) / (np.linalg.norm(emb_pytania) * np.linalg.norm(emb_fragment_1))

### Koniec uzupełniania kodu ###

print(f"Pytanie: '{pytanie}'")
print(f"\nFragment 1: '{chunks[0][:80]}...'")
print(f"  Cosine similarity: {similarity_0:.4f}")
print(f"\nFragment 2: '{chunks[1][:80]}...'")
print(f"  Cosine similarity: {similarity_1:.4f}")
print(f"\nKtóry fragment jest bliższy pytaniu? {'Fragment 1' if similarity_0 > similarity_1 else 'Fragment 2'}")

## 6. Wyszukiwanie semantyczne

Teraz najciekawsza część! Zamieniamy pytanie użytkownika na embedding,
a potem szukamy **najbliższych** fragmentów tekstu.

"Najbliższy" = najwyższe **cosine similarity** (podobieństwo kosinusowe).

To jest *serce* RAG-a — wyszukiwanie semantyczne. Zauważcie, że nie szukamy po słowach kluczowych
(jak w Google), ale po **znaczeniu** tekstu!

In [ ]:
# === Funkcja wyszukiwania semantycznego ===

def search(query, chunks, chunk_embeddings, embed_model, top_k=3):
    """
    Wyszukuje top_k najbardziej pasujących fragmentów do zapytania.
    
    Kroki:
    1. Zamień pytanie na embedding
    2. Oblicz cosine similarity z każdym fragmentem
    3. Zwróć top_k najlepszych
    """
    # 1. Embedding pytania
    query_embedding = embed_model.encode([query])[0]
    
    # 2. Cosine similarity = dot product znormalizowanych wektorów
    #    similarity(A, B) = (A · B) / (||A|| * ||B||)
    similarities = []
    for i, chunk_emb in enumerate(chunk_embeddings):
        dot_product = np.dot(query_embedding, chunk_emb)
        norm_q = np.linalg.norm(query_embedding)
        norm_c = np.linalg.norm(chunk_emb)
        similarity = dot_product / (norm_q * norm_c)
        similarities.append((i, similarity, chunks[i]))
    
    # 3. Sortuj malejąco po similarity
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    return similarities[:top_k]

print("Funkcja wyszukiwania gotowa!")

In [ ]:
# === Testujemy wyszukiwanie! ===

query = "Który prezydent Polski sprawował urząd najkrócej?"

print(f"Pytanie: {query}")
print("="*70)

results = search(query, chunks, chunk_embeddings, embed_model, top_k=3)

print(f"\nZnaleziono {len(results)} najbardziej trafnych fragmentów:\n")

for rank, (idx, score, chunk_text) in enumerate(results, 1):
    print(f"{'─'*70}")
    print(f"  Trafność #{rank} | Fragment nr {idx+1} | Cosine similarity: {score:.4f}")
    print(f"{'─'*70}")
    # Pokazujemy cały fragment — student widzi CO DOKŁADNIE embeddingi znalazły
    print(textwrap.fill(chunk_text, width=70, initial_indent="  ", subsequent_indent="  "))
    print()

In [ ]:
# === Sprawdźmy inne pytania! ===

test_queries = [
    "Kto był najmłodszym prezydentem Polski?",
    "Którzy prezydenci zginęli w trakcie sprawowania urzędu?",
    "Jak długo trwała kadencja Aleksandra Kwaśniewskiego?",
]

for q in test_queries:
    print(f"\n{'='*70}")
    print(f"PYTANIE: {q}")
    print(f"{'='*70}")
    results = search(q, chunks, chunk_embeddings, embed_model, top_k=2)
    for rank, (idx, score, text) in enumerate(results, 1):
        print(f"\n  #{rank} [similarity: {score:.4f}] Fragment {idx+1}:")
        print(textwrap.fill(text[:200], width=70, initial_indent="  ", subsequent_indent="  "))
        if len(text) > 200:
            print("  ...")

## 7. RAG — łączymy wyszukiwanie z LLM-em!

Teraz kluczowy moment. Mamy:
- **Pytanie** użytkownika
- **Fragmenty** tekstu znalezione przez wyszukiwanie semantyczne

Składamy to razem w **prompt** dla LLM-a:

```
Kontekst (znalezione fragmenty):
[Fragment 1]
[Fragment 2]
[Fragment 3]

Pytanie: ...
Odpowiedz WYŁĄCZNIE na podstawie powyższego kontekstu.
```

To jest cały "sekret" RAG-a — dajemy LLM-owi kontekst, którego sam nie ma!

In [ ]:
# === Budowanie prompta RAG ===

def build_rag_prompt(query, retrieved_chunks):
    """
    Buduje prompt RAG z pytaniem i znalezionymi fragmentami.
    """
    context_parts = []
    for rank, (idx, score, text) in enumerate(retrieved_chunks, 1):
        context_parts.append(f"[Fragment {rank} | trafność: {score:.3f}]\n{text}")
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""Na podstawie poniższych fragmentów odpowiedz na pytanie.
Używaj WYŁĄCZNIE informacji z podanych fragmentów.
Jeśli fragmenty nie zawierają odpowiedzi, powiedz że nie wiesz.

=== FRAGMENTY ===
{context}
=== KONIEC FRAGMENTÓW ===

Pytanie: {query}

Odpowiedź:"""
    
    return prompt

# Przetestujmy jak wygląda gotowy prompt
results = search(TEST_QUESTION, chunks, chunk_embeddings, embed_model, top_k=3)
rag_prompt = build_rag_prompt(TEST_QUESTION, results)

print("=== GOTOWY PROMPT RAG (to idzie do LLM-a) ===")
print(rag_prompt)
print(f"\n\nDługość prompta: {len(rag_prompt)} znaków")

### Ćwiczenie 3: Zmodyfikuj prompt RAG

Zmień instrukcję systemową lub prompt użytkownika. Na przykład:
- Każ modelowi odpowiadać w punktach
- Każ modelowi podać źródło (który fragment)
- Zmień język odpowiedzi na angielski

Jak zmiana promptu wpływa na jakość odpowiedzi?

In [ ]:
# === Ćwiczenie 3: Twój własny prompt RAG ===

### Proszę poniżej uzupełnić kod ### (≈ 5 linijek)
# Wskazówka: zmodyfikuj treść prompta — np. dodaj instrukcję "Odpowiedz w 3 punktach"

def moj_rag_prompt(query, retrieved_chunks):
    context_parts = []
    for rank, (idx, score, text) in enumerate(retrieved_chunks, 1):
        context_parts.append(f"[Fragment {rank}]\n{text}")
    context = "\n\n".join(context_parts)
    
    # TUTAJ ZMIEŃ TREŚĆ PROMPTA:
    prompt = f"""Na podstawie poniższych fragmentów odpowiedz na pytanie.
Odpowiedz W TRZECH PUNKTACH. Przy każdym punkcie podaj numer fragmentu źródłowego.

=== FRAGMENTY ===
{context}
=== KONIEC FRAGMENTÓW ===

Pytanie: {query}"""
    return prompt

### Koniec uzupełniania kodu ###

# Testujemy
q = "Co wiesz o prezydencie Lechu Kaczyńskim?"
results = search(q, chunks, chunk_embeddings, embed_model, top_k=3)
moj_prompt = moj_rag_prompt(q, results)

if OLLAMA_URL:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": moj_prompt}],
        temperature=0.1,
    )
    print(f"Pytanie: {q}\n")
    print(response.choices[0].message.content)
else:
    print("LLM niedostępny — ale prompt wygląda tak:")
    print(moj_prompt)

In [ ]:
# === Odpowiedź LLM-a Z RAG-iem ===

if OLLAMA_URL:
    print(f"Pytanie: {TEST_QUESTION}")
    print(f"Model: {MODEL_NAME}")
    print("="*60)
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku, krótko i na temat."},
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.1,
    )
    
    rag_answer = response.choices[0].message.content
    print(f"\nOdpowiedź LLM-a Z RAG-iem:\n")
    print(textwrap.fill(rag_answer, width=80))
else:
    print("LLM niedostępny. Powyżej widać prompt który normalnie poszedłby do modelu.")

## 8. Porównanie: BEZ RAG vs Z RAG

Zobaczmy różnicę na wielu pytaniach!

In [ ]:
# === Pełne porównanie: bez RAG vs z RAG ===

questions = [
    "Kto był najmłodszym prezydentem Polski i ile miał lat?",
    "Który prezydent zginął w katastrofie lotniczej?",
    "Jak długo August Zaleski pełnił funkcję prezydenta?",
    "Kto tymczasowo pełnił obowiązki prezydenta przez 1 dzień?",
]

if OLLAMA_URL:
    for q in questions:
        print(f"\n{'='*70}")
        print(f"PYTANIE: {q}")
        print(f"{'='*70}")
        
        # Bez RAG
        resp_no_rag = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Odpowiadaj krótko i po polsku."},
                {"role": "user", "content": q}
            ],
            temperature=0.1,
        )
        
        # Z RAG
        results = search(q, chunks, chunk_embeddings, embed_model, top_k=3)
        rag_prompt = build_rag_prompt(q, results)
        
        resp_rag = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku, krótko i na temat."},
                {"role": "user", "content": rag_prompt}
            ],
            temperature=0.1,
        )
        
        print(f"\n  BEZ RAG: {resp_no_rag.choices[0].message.content[:200]}")
        print(f"\n  Z RAG:   {resp_rag.choices[0].message.content[:200]}")
        
        # Pokaż użyte fragmenty
        print(f"\n  Użyte fragmenty (top {len(results)}):")
        for rank, (idx, score, text) in enumerate(results, 1):
            print(f"    #{rank} [sim={score:.3f}]: {text[:80]}...")
else:
    print("LLM niedostępny — porównanie wymaga modelu.")

## 9. Zadaj własne pytanie!

Teraz Twoja kolej — wpisz pytanie o prezydentach Polski i zobacz jak działa RAG.

In [ ]:
# === Twoje pytanie! ===

TWOJE_PYTANIE = "Ile państw uznawało rząd RP na uchodźstwie?"  # <-- ZMIEŃ NA SWOJE!


# --- Wyszukiwanie semantyczne ---
print(f"Pytanie: {TWOJE_PYTANIE}")
print("="*70)

results = search(TWOJE_PYTANIE, chunks, chunk_embeddings, embed_model, top_k=3)

print(f"\nZnalezione fragmenty:\n")
for rank, (idx, score, text) in enumerate(results, 1):
    print(f"  #{rank} [cosine similarity: {score:.4f}]:")
    print(textwrap.fill(text, width=70, initial_indent="    ", subsequent_indent="    "))
    print()

# --- RAG ---
if OLLAMA_URL:
    rag_prompt = build_rag_prompt(TWOJE_PYTANIE, results)
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku."},
            {"role": "user", "content": rag_prompt}
        ],
        temperature=0.1,
    )
    
    print(f"{'='*70}")
    print(f"Odpowiedź RAG:\n")
    print(textwrap.fill(response.choices[0].message.content, width=70))
else:
    print("LLM niedostępny — ale powyżej widzisz co wyszukiwanie znalazło!")

## 10. Co pod spodem? Wizualizacja embeddingów

Zobaczmy jak nasze fragmenty i pytanie wyglądają w przestrzeni wektorowej.
Zredukujemy 384-wymiarowe embeddingi do 2D za pomocą **t-SNE** (znane z analizy danych).

In [ ]:
# === Wizualizacja embeddingów ===
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Embedding pytania
query_for_viz = "Który prezydent sprawował urząd najkrócej?"
query_emb = embed_model.encode([query_for_viz])

# Łączymy embeddingi fragmentów + pytania
all_embeddings = np.vstack([chunk_embeddings, query_emb])

# t-SNE redukcja do 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(all_embeddings)-1))
embeddings_2d = tsne.fit_transform(all_embeddings)

# Rysujemy
fig, ax = plt.subplots(1, 1, figsize=(10, 7))

# Fragmenty dokumentu
chunk_points = embeddings_2d[:-1]
ax.scatter(chunk_points[:, 0], chunk_points[:, 1], c='steelblue', s=100, alpha=0.7, label='Fragmenty dokumentu')

# Etykiety fragmentów
for i, (x, y) in enumerate(chunk_points):
    ax.annotate(f'F{i+1}', (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)

# Pytanie
query_point = embeddings_2d[-1]
ax.scatter(query_point[0], query_point[1], c='red', s=200, marker='*', label=f'Pytanie', zorder=5)
ax.annotate(f'Q: {query_for_viz[:40]}...', (query_point[0], query_point[1]),
            textcoords="offset points", xytext=(10, 10), fontsize=9, color='red', fontweight='bold')

# Zaznaczmy top-3 najbliższe fragmenty
results = search(query_for_viz, chunks, chunk_embeddings, embed_model, top_k=3)
for rank, (idx, score, _) in enumerate(results):
    x, y = chunk_points[idx]
    ax.scatter(x, y, c='orange', s=200, alpha=0.5, edgecolors='red', linewidth=2)
    ax.annotate(f'TOP {rank+1}\n(sim={score:.3f})', (x, y),
                textcoords="offset points", xytext=(-15, -20), fontsize=8, color='darkorange')

ax.set_title('Embeddingi fragmentów dokumentu + pytanie (t-SNE 2D)', fontsize=13)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nFragmenty najbliżej pytania (pomarańczowe) to te, które trafiają do RAG prompta!")

## Podsumowanie

### Co zrobiliśmy?

1. **Wczytaliśmy dokument** i podzielili go na fragmenty (chunks)
2. **Zamienili fragmenty na embeddingi** (wektory liczbowe) — podobnie jak Word2Vec, ale dla całych zdań
3. **Wyszukaliśmy semantycznie** — pytanie też zamieniamy na embedding i szukamy najbliższych fragmentów
4. **Zbudowaliśmy prompt RAG** — wklejamy znalezione fragmenty + pytanie i wysyłamy do LLM-a
5. **LLM odpowiedział poprawnie** — bo miał kontekst, którego sam nie znał!

### RAG w produkcji

W prawdziwych zastosowaniach zamiast naszego prostego kodu używa się:
- **Wektorowych baz danych** (Pinecone, Weaviate, ChromaDB, FAISS) zamiast `numpy`
- **Frameworków** (LangChain, LlamaIndex) do orkiestracji
- **Lepszych strategii chunkingu** (recursive, semantic chunking)
- **Re-rankingu** (dodatkowy model który ponownie ocenia trafność)
- **Hybrydowego wyszukiwania** (embeddingi + klasyczny BM25)

Ale **zasada jest dokładnie taka sama** jak to co zrobiliśmy na tych zajęciach!

### Kiedy RAG, a kiedy fine-tuning?

| | RAG | Fine-tuning (SFT) |
|---|---|---|
| **Kiedy?** | Dane się zmieniają, trzeba cytować źródła | Chcemy zmienić "styl" lub "wiedzę" modelu na stałe |
| **Koszt** | Tani (tylko inference) | Drogi (trening na GPU) |
| **Aktualność** | Zawsze aktualne (aktualizujesz dokumenty) | Wymaga ponownego treningu |
| **Halucynacje** | Mniejsze (model ma kontekst) | Możliwe (model "pamięta" błędnie) |